# 1.2 - Data Engineering - Gauge Summary and Valid Days

Este notebook calcula a consistência temporal (dias válidos por ano) de cada estação e sumariza as propriedades anuais dos eventos de chuva (`yearly_rainfall`, `rainfall_event`, `rainfall_intensity`, `rainfall_duration`), separando as estações de referência para calibração.

**Critérios de Consistência:**
- Mínimo de 300 dias com dados válidos no ano;
- Precipitação anual realista: entre 300 mm e 3000 mm.

In [ ]:
%config Completer.use_jedi = False
%load_ext autoreload
%autoreload 2

import sys, os, warnings
from pathlib import Path
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd() if Path.cwd().name == 'src' else Path.cwd() / 'src'
sys.path.insert(0, str(PROJECT_ROOT.parent))

from src.config import ROOT, ANNUAL_RAIN_MIN, ANNUAL_RAIN_MAX, MIN_DAY_COUNT, MITS_MINUTES
from src.data.loader import load_gauge_info, load_rainfall_data
from src.events.detection import compute_event_statistics, count_valid_days
from src.utils.logger import save_result

RESULTS_DIR = ROOT / 'src' / '5 - Results'

## 1. Carregamento dos Eventos Discretizados

In [ ]:
events_path = RESULTS_DIR / 'events_table.parquet'
df_events = pd.read_parquet(events_path)
print(f'Eventos carregados: {len(df_events):,}')
df_info = load_gauge_info()
df_events.head()

## 2. Consolidação das Propriedades Anuais por Estação e MIT

In [ ]:
gauge_info_dict = df_info.set_index('gauge_code').to_dict('index')
summary_records = []

grouped = df_events.groupby(['gauge_code', 'year', 'mit_minutes'])
for (code_val, year_val, mit_val), group in grouped:
    info = gauge_info_dict.get(str(code_val))
    if not info:
        continue
    info_series = pd.Series(info)
    info_series['gauge_code'] = str(code_val)
    stats = compute_event_statistics(group, info_series, mit_val)
    if stats:
        stats['year'] = year_val
        summary_records.append(stats)

df_annual_props = pd.DataFrame(summary_records)
print(f'Total de resumos anuais por estação/MIT: {len(df_annual_props):,}')
df_annual_props.head()

## 3. Identificação das Estações de Referência (Baliza)

In [ ]:
df_annual_props['is_reference'] = (
    (df_annual_props['yearly_rainfall'] >= ANNUAL_RAIN_MIN) &
    (df_annual_props['yearly_rainfall'] <= ANNUAL_RAIN_MAX)
)
print('Estações de Referência por MIT:')
print(df_annual_props.groupby('mit_minutes')['is_reference'].value_counts())

out_path = save_result(df_annual_props, 'annual_gauge_properties.parquet', subdir='5 - Results')
print('Propriedades anuais salvas em:', out_path)